# Project 00 — Catalogue Feasibility Audit

## Scientific objective

Determine whether existing open-cluster tidal-tail catalogues contain enough reliable outer-tail members with independent kinematic or spectroscopic information to support a paper-level test of Galactic-bar-sensitive tidal-tail predictions.

## Scope of this notebook

This notebook performs the feasibility stage:

1. retrieve and preserve the published catalogue data;
2. inspect catalogue schema and provenance;
3. inventory clusters and literature catalogues;
4. identify reliable leading/trailing-tail candidates;
5. quantify the available radial-velocity information;
6. construct a cluster-level feasibility table for target selection.

No Galactic-potential inference is performed at this stage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.table import Table
from astroquery.vizier import Vizier

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Imports OK")

Imports OK


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"
RESULTS_FIGURES = PROJECT_ROOT / "results" / "figures"

print("Project root:", PROJECT_ROOT)

assert PROJECT_ROOT.name == "gaia-tidal-tail-dynamics"
assert DATA_RAW.exists()

print("Project paths OK")

Project root: /Users/liors/Documents/research/gaia-tidal-tail-dynamics
Project paths OK


## 1. Catalogue acquisition

We begin with the published Jadhav et al. open-cluster tidal-tail compilation available through VizieR as `J/A+A/704/A50`.

At this stage the catalogue is used as a **provenance and reliability layer**, rather than as a ground-truth membership catalogue.

The remote catalogue is first queried with a small row limit to verify its schema before the complete tables are acquired.

In [3]:
Vizier.TIMEOUT = 300
Vizier.ROW_LIMIT = 10

CATALOGUE_ID = "J/A+A/704/A50"

viz_probe = Vizier(
    columns=["*"],
    row_limit=10,
)

probe = viz_probe.get_catalogs(CATALOGUE_ID)

print(f"Number of tables returned: {len(probe)}")

for key in probe.keys():
    print(key, len(probe[key]))

Number of tables returned: 2
J/A+A/704/A50/clusters 10
J/A+A/704/A50/sources 10


In [4]:
for key in probe.keys():
    table = probe[key]

    print("=" * 80)
    print("TABLE:", key)
    print("ROWS RETURNED:", len(table))
    print("COLUMNS:")
    print(table.colnames)

TABLE: J/A+A/704/A50/clusters
ROWS RETURNED: 10
COLUMNS:
['Cluster', 'Ref', 'ClusterID', 'RA_ICRS', 'DE_ICRS', 'dist50', 'logAge50', 'pmRA', 'pmDE', 'RV', 'fNoPlxI', 'fxyExt', 'fxyShape', 'fxyTor', 'fskyExt', 'fskyTor', 'fall', 'Grade', 'Nall', 'NLead', 'NCluster', 'NTrail', 'NrefAll', 'NrefLead', 'NrefCluster', 'NrefTrail', 'NbinAll', 'NbinLead', 'NbinCluster', 'NbinTrail', 'BFall', 'e_BFall', 'BFLead', 'e_BFLead', 'BFCluster', 'e_BFCluster', 'BFTrail', 'e_BFTrail', 'spanAll', 'spanLead', 'spanTrail', 'SimbadName', '_RA.icrs', '_DE.icrs']
TABLE: J/A+A/704/A50/sources
ROWS RETURNED: 10
COLUMNS:
['GaiaDR3', 'Cluster', 'Ref', 'ClusterID', 'Grade', 'Class', 'RA_ICRS', 'DE_ICRS', 'rmedgeo', 'pmRA', 'pmDE', 'RV', 'Gmag', 'BPmag', 'RPmag', 'BP-RP', 'GMAG', 'r3d', 'v3d', 'rsky', 'pmR', 'e_pmR', 'pmT', 'e_pmT', 'deltaRVGC', 'deltaphiGC', 'distFromO', 'distAlongO', 'cmdDist', 'q', 'e_q', 'MassSysMLR', 'MassSys', 'e_MassSys', 'MassA', 'e_MassA', 'MassB', 'e_MassB', '_RA.icrs', '_DE.icrs']


## 2. Immutable raw-data snapshot

After schema verification, the complete published tables are retrieved once and stored locally under `data/raw/`.

These files are treated as immutable source data. Subsequent analysis should read the local snapshot rather than repeatedly querying the remote service.

In [6]:
Vizier.TIMEOUT = 600
Vizier.ROW_LIMIT = -1

viz_full = Vizier(
    columns=["*"],
    row_limit=-1,
)

full = viz_full.get_catalogs(CATALOGUE_ID)

print("Number of tables:", len(full))
print("Actual keys:", full.keys())

for i, table in enumerate(full):
    print(
        f"Table {i}: "
        f"rows={len(table)}, "
        f"columns={len(table.colnames)}"
    )

Number of tables: 1
Actual keys: ['J/A+A/704/A50/sources']
Table 0: rows=57981, columns=40


In [8]:
sources = full["J/A+A/704/A50/sources"]

print("Sources table:", len(sources))
print("Number of columns:", len(sources.colnames))

print("\nColumns:")
print(sources.colnames)

assert len(sources) == 57981
assert "GaiaDR3" in sources.colnames
assert "Cluster" in sources.colnames
assert "Grade" in sources.colnames
assert "Class" in sources.colnames

print("\nFull source catalogue validation: PASSED")

Sources table: 57981
Number of columns: 40

Columns:
['GaiaDR3', 'Cluster', 'Ref', 'ClusterID', 'Grade', 'Class', 'RA_ICRS', 'DE_ICRS', 'rmedgeo', 'pmRA', 'pmDE', 'RV', 'Gmag', 'BPmag', 'RPmag', 'BP-RP', 'GMAG', 'r3d', 'v3d', 'rsky', 'pmR', 'e_pmR', 'pmT', 'e_pmT', 'deltaRVGC', 'deltaphiGC', 'distFromO', 'distAlongO', 'cmdDist', 'q', 'e_q', 'MassSysMLR', 'MassSys', 'e_MassSys', 'MassA', 'e_MassA', 'MassB', 'e_MassB', '_RA.icrs', '_DE.icrs']

Full source catalogue validation: PASSED


In [9]:
sources_path = DATA_RAW / "jadhav2025_sources.ecsv"

if not sources_path.exists():
    sources.write(
        sources_path,
        format="ascii.ecsv"
    )
    print("Saved:", sources_path)
else:
    print("Already exists:", sources_path)

Saved: /Users/liors/Documents/research/gaia-tidal-tail-dynamics/data/raw/jadhav2025_sources.ecsv


In [10]:
sources_local = Table.read(
    sources_path,
    format="ascii.ecsv"
)

print("Local sources:", len(sources_local))

assert len(sources_local) == 57981
assert sources_local.colnames == sources.colnames

print("Raw snapshot verification: PASSED")

Local sources: 57981
Raw snapshot verification: PASSED


In [11]:
df = sources_local.to_pandas()

print("Shape:", df.shape)
display(df.head())

Shape: (57981, 40)


,GaiaDR3,Cluster,Ref,ClusterID,Grade,Class,RA_ICRS,DE_ICRS,rmedgeo,pmRA,pmDE,RV,Gmag,BPmag,RPmag,BP-RP,GMAG,r3d,v3d,rsky,pmR,e_pmR,pmT,e_pmT,deltaRVGC,deltaphiGC,distFromO,distAlongO,cmdDist,q,e_q,MassSysMLR,MassSys,e_MassSys,MassA,e_MassA,MassB,e_MassB,_RA.icrs,_DE.icrs
0,2050809775326179584,ASCC_101,Bhattacharya2022,0,B,T,288.178653,36.694359,368.3396,0.8882,1.4151,NaN,17.556200,18.8691,16.4219,2.4472,9.7250,29.059999,NaN,0.3601,0.0203,0.0533,0.1305,0.1119,NaN,0.1903,12.45,-26.420000,0.0043,0.0,0.00,0.4147,0.4410,0.0965,0.4410,0.0965,0.0000,0.0000,288.178648,36.694353
1,2050770300286488960,ASCC_101,Bhattacharya2022,0,B,T,288.087589,36.436499,374.5052,1.0991,0.8689,-14.1317,11.689300,11.9123,11.2397,0.6726,3.8220,22.840000,4.09,0.2207,-0.2828,0.0179,0.4356,0.0474,-2.0346,0.1532,8.55,-21.049999,0.0057,0.6,0.12,1.2654,1.9813,0.2940,1.2383,0.1838,0.7430,0.1103,288.087583,36.436495
2,2050715286043161216,ASCC_101,Bhattacharya2022,0,B,T,288.422649,36.248052,374.5173,1.0794,1.4331,NaN,18.400101,20.1710,17.1695,3.0014,10.5327,22.790001,NaN,0.1283,-0.0624,0.0614,0.0819,0.1717,NaN,0.1511,9.03,-21.049999,0.0028,0.5,0.11,0.3560,0.5040,0.0830,0.3360,0.0553,0.1680,0.0277,288.422643,36.248046
3,2050573311606246912,ASCC_101,Bhattacharya2022,0,B,T,287.575011,35.435401,378.1879,0.8380,1.0010,NaN,18.155899,19.5864,16.9769,2.6094,10.2674,20.540001,NaN,1.1150,0.1900,0.1741,0.0587,0.0242,NaN,0.1461,2.44,-20.600000,0.0065,0.0,0.00,0.3743,0.3820,0.0874,0.3820,0.0874,0.0000,0.0000,287.575006,35.435397
4,2050509097557530624,ASCC_101,Bhattacharya2022,0,B,T,288.618800,35.958059,375.9113,1.4791,1.2507,-17.3185,12.395900,12.7228,11.9009,0.8219,4.5204,21.610001,1.09,0.4588,0.2258,0.0358,0.3478,0.1066,-0.0589,0.1426,8.61,-19.700001,0.0050,0.5,0.11,1.1089,1.6481,0.2579,1.0987,0.1719,0.5494,0.0860,288.618792,35.958053


In [12]:
print("Unique Gaia DR3 sources:", df["GaiaDR3"].nunique())
print("Unique clusters:", df["Cluster"].nunique())
print("Unique references:", df["Ref"].nunique())

print("\nGrade values:")
print(df["Grade"].value_counts(dropna=False))

print("\nClass values:")
print(df["Class"].value_counts(dropna=False))

Unique Gaia DR3 sources: 33125
Unique clusters: 58
Unique references: 17

Grade values:
Grade
S    26530
B    21565
G     9886
Name: count, dtype: int64

Class values:
Class
C    38872
L     9726
T     9383
Name: count, dtype: int64


In [13]:
grade_class = pd.crosstab(
    df["Grade"],
    df["Class"],
    margins=True
)

display(grade_class)

Class,C,L,T,All
Grade,,,,
B,14402,3537,3626,21565
G,6205,2134,1547,9886
S,18265,4055,4210,26530
All,38872,9726,9383,57981


In [14]:
reliable_tail = df[
    df["Grade"].isin(["G", "S"])
    & df["Class"].isin(["L", "T"])
].copy()

print("Gold/Silver tail records:", len(reliable_tail))
print("Unique Gold/Silver tail stars:", reliable_tail["GaiaDR3"].nunique())
print("Clusters represented:", reliable_tail["Cluster"].nunique())

Gold/Silver tail records: 11946
Unique Gold/Silver tail stars: 8594
Clusters represented: 40


## 3. Cluster-level feasibility inventory

The reliable-tail sample contains Gold/Silver catalogue records classified
as leading or trailing-tail members.

Because the same Gaia DR3 source may appear in multiple literature
catalogues, both record counts and unique-source counts are tracked.
Catalogue multiplicity is retained as provenance information rather than
interpreted directly as independent evidence.

In [15]:
cluster_inventory = (
    reliable_tail
    .groupby("Cluster")
    .agg(
        tail_records=("GaiaDR3", "size"),
        unique_tail_stars=("GaiaDR3", "nunique"),
        n_references=("Ref", "nunique"),
    )
)

leading = (
    reliable_tail[reliable_tail["Class"] == "L"]
    .groupby("Cluster")["GaiaDR3"]
    .nunique()
    .rename("unique_leading")
)

trailing = (
    reliable_tail[reliable_tail["Class"] == "T"]
    .groupby("Cluster")["GaiaDR3"]
    .nunique()
    .rename("unique_trailing")
)

cluster_inventory = (
    cluster_inventory
    .join(leading)
    .join(trailing)
    .fillna(0)
)

cluster_inventory[
    ["unique_leading", "unique_trailing"]
] = cluster_inventory[
    ["unique_leading", "unique_trailing"]
].astype(int)

cluster_inventory = (
    cluster_inventory
    .sort_values("unique_tail_stars", ascending=False)
)

display(cluster_inventory.head(20))

,tail_records,unique_tail_stars,n_references,unique_leading,unique_trailing
Cluster,,,,,
Melotte_25,2565,1068,7,606,462
NGC_2516,812,812,1,478,334
Melotte_22,842,664,4,463,201
Melotte_20,723,606,3,251,355
NGC_2632,818,522,2,333,189
Stock_2,676,516,2,184,332
NGC_3532,483,483,1,182,301
Theia_517,780,424,5,155,269
NGC_752,338,338,1,179,159


In [16]:
reliable_tail["has_rv"] = reliable_tail["RV"].notna()

rv_inventory = (
    reliable_tail
    .groupby("Cluster")
    .agg(
        tail_records_with_rv=("has_rv", "sum"),
    )
)

unique_rv = (
    reliable_tail[reliable_tail["has_rv"]]
    .groupby("Cluster")["GaiaDR3"]
    .nunique()
    .rename("unique_tail_stars_with_rv")
)

cluster_inventory = (
    cluster_inventory
    .join(rv_inventory)
    .join(unique_rv)
    .fillna(0)
)

cluster_inventory["unique_tail_stars_with_rv"] = (
    cluster_inventory["unique_tail_stars_with_rv"].astype(int)
)

cluster_inventory["rv_fraction"] = (
    cluster_inventory["unique_tail_stars_with_rv"]
    / cluster_inventory["unique_tail_stars"]
)

display(
    cluster_inventory
    .sort_values("unique_tail_stars", ascending=False)
    .head(20)
)

,tail_records,unique_tail_stars,n_references,unique_leading,unique_trailing,tail_records_with_rv,unique_tail_stars_with_rv,rv_fraction
Cluster,,,,,,,,
Melotte_25,2565,1068,7,606,462,1307,499,0.467228
NGC_2516,812,812,1,478,334,208,208,0.256158
Melotte_22,842,664,4,463,201,285,216,0.325301
Melotte_20,723,606,3,251,355,298,248,0.409241
NGC_2632,818,522,2,333,189,230,142,0.272031
Stock_2,676,516,2,184,332,382,292,0.565891
NGC_3532,483,483,1,182,301,197,197,0.407867
Theia_517,780,424,5,155,269,236,118,0.278302
NGC_752,338,338,1,179,159,106,106,0.313609


In [18]:
target_aliases = {
    "Hyades": "Melotte_25",
    "Praesepe": "NGC_2632",
    "Stock 2": "Stock_2",
    "IC 4756": "IC_4756",
}

target_inventory = cluster_inventory.loc[
    list(target_aliases.values())
].copy()

target_inventory.insert(
    0,
    "common_name",
    list(target_aliases.keys())
)

display(target_inventory)

,common_name,tail_records,unique_tail_stars,n_references,unique_leading,unique_trailing,tail_records_with_rv,unique_tail_stars_with_rv,rv_fraction
Cluster,,,,,,,,,
Melotte_25,Hyades,2565,1068,7,606,462,1307,499,0.467228
NGC_2632,Praesepe,818,522,2,333,189,230,142,0.272031
Stock_2,Stock 2,676,516,2,184,332,382,292,0.565891
IC_4756,IC 4756,135,135,1,88,47,67,67,0.496296


## 4. Radial-velocity provenance audit

The presence of an `RV` value is used only as a preliminary feasibility
indicator. It is not yet interpreted as independent spectroscopic
validation.

Before using RV coverage scientifically, we inspect its distribution across
literature references and duplicated Gaia DR3 sources.

In [19]:
rv_by_ref = (
    reliable_tail
    .assign(has_rv=reliable_tail["RV"].notna())
    .groupby("Ref")
    .agg(
        records=("GaiaDR3", "size"),
        unique_stars=("GaiaDR3", "nunique"),
        records_with_rv=("has_rv", "sum"),
    )
)

rv_by_ref["record_rv_fraction"] = (
    rv_by_ref["records_with_rv"] / rv_by_ref["records"]
)

display(
    rv_by_ref.sort_values(
        "unique_stars",
        ascending=False
    )
)

,records,unique_stars,records_with_rv,record_rv_fraction
Ref,,,,
Risbud2025,3666,3666,1542,0.420622
Kos2024,2692,2691,1250,0.464339
Meingast2021,1734,1734,545,0.314302
Roser2019,836,836,387,0.462919
Jerabkova2021M1,624,624,264,0.423077
Bhattacharya2022,459,459,144,0.313725
Oh2020,443,443,252,0.568849
Jerabkova2021M5,431,431,162,0.375870
Boffin2022,338,338,106,0.313609


In [20]:
for common_name, catalogue_name in target_aliases.items():

    subset = reliable_tail[
        reliable_tail["Cluster"] == catalogue_name
    ]

    print("=" * 70)
    print(common_name, f"({catalogue_name})")

    summary = (
        subset
        .assign(has_rv=subset["RV"].notna())
        .groupby("Ref")
        .agg(
            records=("GaiaDR3", "size"),
            unique_stars=("GaiaDR3", "nunique"),
            rv_records=("has_rv", "sum"),
        )
        .sort_values("unique_stars", ascending=False)
    )

    display(summary)

Hyades (Melotte_25)


,records,unique_stars,rv_records
Ref,,,
Jerabkova2021M1,624,624,264
Risbud2025,537,537,285
Roser2019,448,448,263
Oh2020,443,443,252
Jerabkova2021M5,431,431,162
Meingast2019,78,78,77
Kos2024,4,4,4


Praesepe (NGC_2632)


,records,unique_stars,rv_records
Ref,,,
Risbud2025,430,430,106
Roser2019,388,388,124


Stock 2 (Stock_2)


,records,unique_stars,rv_records
Ref,,,
Risbud2025,442,442,256
Kos2024,234,234,126


IC 4756 (IC_4756)


,records,unique_stars,rv_records
Ref,,,
Kos2024,135,135,67


## 5. Cluster-centre metadata

Physical tail extent requires a consistent cluster-centre position and
distance. The cluster metadata table is therefore retrieved separately
before any cluster-centric spatial separation is calculated.

In [22]:
from astroquery.vizier import Vizier

Vizier.VIZIER_SERVER = "vizier.cfa.harvard.edu"

cluster_query = Vizier(
    columns=["*"],
    row_limit=-1,
)

cluster_result = cluster_query.get_catalogs(
    "J/A+A/704/A50/clusters"
)

print("Returned tables:", cluster_result.keys())

for key in cluster_result.keys():
    print(key, len(cluster_result[key]))

Returned tables: ['J/A+A/704/A50/clusters']
J/A+A/704/A50/clusters 122


In [23]:
clusters = cluster_result[0]

print("Rows:", len(clusters))
print("Columns:")
print(clusters.colnames)

display(clusters[:10])

Rows: 122
Columns:
['Cluster', 'Ref', 'ClusterID', 'RA_ICRS', 'DE_ICRS', 'dist50', 'logAge50', 'pmRA', 'pmDE', 'RV', 'fNoPlxI', 'fxyExt', 'fxyShape', 'fxyTor', 'fskyExt', 'fskyTor', 'fall', 'Grade', 'Nall', 'NLead', 'NCluster', 'NTrail', 'NrefAll', 'NrefLead', 'NrefCluster', 'NrefTrail', 'NbinAll', 'NbinLead', 'NbinCluster', 'NbinTrail', 'BFall', 'e_BFall', 'BFLead', 'e_BFLead', 'BFCluster', 'e_BFCluster', 'BFTrail', 'e_BFTrail', 'spanAll', 'spanLead', 'spanTrail', 'SimbadName', '_RA.icrs', '_DE.icrs']


Cluster,Ref,ClusterID,RA_ICRS,DE_ICRS,dist50,logAge50,pmRA,pmDE,RV,fNoPlxI,fxyExt,fxyShape,fxyTor,fskyExt,fskyTor,fall,Grade,Nall,NLead,NCluster,NTrail,NrefAll,NrefLead,NrefCluster,NrefTrail,NbinAll,NbinLead,NbinCluster,NbinTrail,BFall,e_BFall,BFLead,e_BFLead,BFCluster,e_BFCluster,BFTrail,e_BFTrail,spanAll,spanLead,spanTrail,SimbadName,_RA.icrs,_DE.icrs
,,,deg,deg,pc,log(yr),mas / yr,mas / yr,km / s,,,,,,,,,,,,,,,,,,,,,,,,,,,,,pc,pc,pc,,deg,deg
str13,str16,int16,float64,float64,float64,float64,float64,float64,float64,uint8,uint8,uint8,uint8,uint8,uint8,uint8,str1,int16,int16,int16,int16,int16,int16,int16,int16,int16,int16,int16,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str15,float64,float64
ASCC_101,Bhattacharya2022,0,288.34468256,36.35990257,397.29552501,8.27605438,0.96745283,1.28139139,-18.11557240,0,1,1,0,1,0,3,B,103,12,72,19,91,9,64,18,22,4,13,5,0.242,0.057,0.444,0.267,0.203,0.062,0.278,0.140,41.140,20.095,21.045,[KPR2005] 101,288.34467722,36.35989687
ASCC_101,Kos2024,1,288.34468256,36.35990257,397.29552501,8.27605438,0.96745283,1.28139139,-18.11557240,0,1,1,0,1,0,3,B,90,13,60,17,73,10,49,14,15,2,9,4,0.205,0.058,0.200,0.155,0.184,0.067,0.286,0.162,69.738,36.857,32.881,[KPR2005] 101,288.34467722,36.35989687
ASCC_101,Risbud2025,2,288.34468256,36.35990257,397.29552501,8.27605438,0.96745283,1.28139139,-18.11557240,1,1,0,1,0,1,4,S,144,11,61,72,129,10,54,65,20,2,7,11,0.155,0.037,0.200,0.155,0.130,0.052,0.169,0.055,134.409,26.927,107.482,[KPR2005] 101,288.34467722,36.35989687
ASCC_99,Kos2024,3,282.17181947,-18.38406542,298.96316195,7.94974232,5.24745277,-1.30712533,-29.85613033,1,1,0,1,1,1,5,S,188,37,107,44,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,276.083,86.039,190.044,[KPR2005] 99,282.17179489,-18.38405961
Alessi_24,Bhattacharya2022,4,260.97843601,-62.81755799,476.72721464,7.99862313,-0.42996834,-8.96254860,9.59254084,0,1,0,0,1,0,2,B,149,14,120,15,135,12,112,11,19,2,15,2,0.141,0.034,0.167,0.127,0.134,0.037,0.182,0.140,36.760,15.858,20.902,Cl Alessi 24,260.97844019,-62.81751816
Alessi_3,Risbud2025,5,109.18670262,-46.68028150,276.57916812,8.79626179,-9.79326553,11.91590602,0.31662029,0,1,0,0,1,1,3,B,233,91,110,32,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,176.540,121.729,54.810,Cl Alessi 3,109.18676606,-46.68033446
Alessi_3,Pang2022,6,109.18670262,-46.68028150,276.57916812,8.79626179,-9.79326553,11.91590602,0.31662029,1,1,1,1,1,0,5,S,165,27,118,20,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,47.939,22.797,25.143,Cl Alessi 3,109.18676606,-46.68033446
Alessi_3,Kos2024,7,109.18670262,-46.68028150,276.57916812,8.79626179,-9.79326553,11.91590602,0.31662029,0,1,1,0,1,0,3,B,145,31,93,21,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,112.986,63.404,49.583,Cl Alessi 3,109.18676606,-46.68033446


In [24]:
clusters_df = clusters.to_pandas()

target_cluster_metadata = clusters_df[
    clusters_df["Cluster"].isin([
        "Melotte_25",
        "NGC_2632",
        "Stock_2",
        "IC_4756",
    ])
].copy()

display(target_cluster_metadata)

,Cluster,Ref,ClusterID,RA_ICRS,DE_ICRS,dist50,logAge50,pmRA,pmDE,RV,fNoPlxI,fxyExt,fxyShape,fxyTor,fskyExt,fskyTor,fall,Grade,Nall,NLead,NCluster,NTrail,NrefAll,NrefLead,NrefCluster,NrefTrail,NbinAll,NbinLead,NbinCluster,NbinTrail,BFall,e_BFall,BFLead,e_BFLead,BFCluster,e_BFCluster,BFTrail,e_BFTrail,spanAll,spanLead,spanTrail,SimbadName,_RA.icrs,_DE.icrs
32,IC_4756,Bhattacharya2022,32,279.627468,5.437089,466.754053,8.922665,1.284348,-4.968163,-24.879046,0,0,0,1,1,0,2,B,451,80,340,31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.471001,31.186001,20.285000,IC_4756,279.627463,5.437112
33,IC_4756,Kos2024,33,279.627468,5.437089,466.754053,8.922665,1.284348,-4.968163,-24.879046,0,1,1,1,1,0,4,S,431,88,296,47,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,135.908997,87.484001,48.424999,IC_4756,279.627463,5.437112
34,IC_4756,Pang2022,34,279.627468,5.437089,466.754053,8.922665,1.284348,-4.968163,-24.879046,0,1,0,1,1,0,3,B,497,84,345,68,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.011002,33.619999,28.391001,IC_4756,279.627463,5.437112
51,Melotte_25,Jerabkova2021M5,51,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,1,1,1,6,G,862,293,431,138,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,868.335999,434.496002,433.839996,Cl Melotte 25,66.708164,16.080925
52,Melotte_25,Jerabkova2021M1,52,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,1,1,1,6,G,1109,384,485,240,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,881.960022,447.230011,434.730011,Cl Melotte 25,66.708164,16.080925
53,Melotte_25,Meingast2019,53,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,1,1,1,6,G,238,41,160,37,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.625000,100.491997,80.133003,Cl Melotte 25,66.708164,16.080925
54,Melotte_25,Oh2020,54,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,1,1,1,6,G,1002,257,559,186,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,219.591003,134.634003,84.957001,Cl Melotte 25,66.708164,16.080925
55,Melotte_25,Risbud2025,55,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,1,1,1,6,G,1071,311,534,226,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,208.332993,136.757996,71.575996,Cl Melotte 25,66.708164,16.080925
56,Melotte_25,Kos2024,56,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,0,1,0,4,S,439,0,435,4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.440001,0.000000,9.518000,Cl Melotte 25,66.708164,16.080925
57,Melotte_25,Roser2019,57,66.708646,16.080798,47.190660,8.761053,104.135545,-28.732309,39.263498,1,1,1,1,1,1,6,G,979,252,531,196,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,213.957993,134.841003,79.115997,Cl Melotte 25,66.708164,16.080925


## 6. Three-dimensional tail extent

Cluster-centric three-dimensional separations are estimated from the
catalogued sky positions and geometric distances.

For each cluster, a single centre is constructed after verifying that the
centre coordinates and distance are consistent across literature-reference
rows.

These separations are used only as a feasibility diagnostic at this stage;
they are not yet interpreted as dynamical tidal radii.

In [25]:
centre_check = (
    clusters_df
    .groupby("Cluster")
    .agg(
        n_rows=("Cluster", "size"),
        n_ra=("RA_ICRS", "nunique"),
        n_dec=("DE_ICRS", "nunique"),
        n_dist=("dist50", "nunique"),
    )
)

problem_centres = centre_check[
    (centre_check["n_ra"] > 1)
    | (centre_check["n_dec"] > 1)
    | (centre_check["n_dist"] > 1)
]

print("Clusters with inconsistent centre metadata:")
display(problem_centres)

print("Number:", len(problem_centres))

Clusters with inconsistent centre metadata:


,n_rows,n_ra,n_dec,n_dist
Cluster,,,,


Number: 0


In [26]:
cluster_centres = (
    clusters_df[
        ["Cluster", "RA_ICRS", "DE_ICRS", "dist50"]
    ]
    .drop_duplicates(subset="Cluster")
    .set_index("Cluster")
)

print("Unique cluster centres:", len(cluster_centres))
display(
    cluster_centres.loc[
        ["Melotte_25", "NGC_2632", "Stock_2", "IC_4756"]
    ]
)

Unique cluster centres: 58


,RA_ICRS,DE_ICRS,dist50
Cluster,,,
Melotte_25,66.708646,16.080798,47.190660
NGC_2632,130.088104,19.665873,183.491092
Stock_2,33.850043,59.577009,370.246233
IC_4756,279.627468,5.437089,466.754053


In [27]:
def spherical_to_cartesian(ra_deg, dec_deg, distance_pc):
    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    x = distance_pc * np.cos(dec) * np.cos(ra)
    y = distance_pc * np.cos(dec) * np.sin(ra)
    z = distance_pc * np.sin(dec)

    return x, y, z

In [28]:
tail_spatial = reliable_tail.copy()

centre_lookup = cluster_centres.reset_index().rename(
    columns={
        "RA_ICRS": "cluster_ra",
        "DE_ICRS": "cluster_dec",
        "dist50": "cluster_distance",
    }
)

tail_spatial = tail_spatial.merge(
    centre_lookup,
    on="Cluster",
    how="left",
    validate="many_to_one",
)

print("Rows:", len(tail_spatial))
print(
    "Missing cluster centres:",
    tail_spatial["cluster_distance"].isna().sum()
)

Rows: 11946
Missing cluster centres: 0


In [29]:
sx, sy, sz = spherical_to_cartesian(
    tail_spatial["RA_ICRS"].to_numpy(),
    tail_spatial["DE_ICRS"].to_numpy(),
    tail_spatial["rmedgeo"].to_numpy(),
)

cx, cy, cz = spherical_to_cartesian(
    tail_spatial["cluster_ra"].to_numpy(),
    tail_spatial["cluster_dec"].to_numpy(),
    tail_spatial["cluster_distance"].to_numpy(),
)

tail_spatial["r_cluster_pc"] = np.sqrt(
    (sx - cx)**2
    + (sy - cy)**2
    + (sz - cz)**2
)

print(
    tail_spatial["r_cluster_pc"]
    .describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

count    11946.000000
mean        54.538543
std         58.500437
min          9.960089
10%         15.245605
25%         21.787830
50%         36.343242
75%         66.176568
90%        106.812544
95%        143.192658
99%        335.217762
max        867.224140
Name: r_cluster_pc, dtype: float64


In [30]:
target_rows = []

for common_name, catalogue_name in target_aliases.items():

    sub = tail_spatial[
        tail_spatial["Cluster"] == catalogue_name
    ].copy()

    # One row per Gaia source for spatial feasibility counts.
    sub_unique = (
        sub.sort_values("has_rv", ascending=False)
        .drop_duplicates("GaiaDR3")
    )

    rv = sub_unique[sub_unique["has_rv"]]

    target_rows.append({
        "target": common_name,
        "tail_stars": sub_unique["GaiaDR3"].nunique(),
        "rv_stars": rv["GaiaDR3"].nunique(),
        "rv_r_gt_10pc": rv.loc[
            rv["r_cluster_pc"] > 10, "GaiaDR3"
        ].nunique(),
        "rv_r_gt_20pc": rv.loc[
            rv["r_cluster_pc"] > 20, "GaiaDR3"
        ].nunique(),
        "rv_r_gt_50pc": rv.loc[
            rv["r_cluster_pc"] > 50, "GaiaDR3"
        ].nunique(),
        "rv_r_gt_100pc": rv.loc[
            rv["r_cluster_pc"] > 100, "GaiaDR3"
        ].nunique(),
        "max_r_pc": rv["r_cluster_pc"].max(),
    })

outer_rv_inventory = (
    pd.DataFrame(target_rows)
    .set_index("target")
)

display(outer_rv_inventory)

,tail_stars,rv_stars,rv_r_gt_10pc,rv_r_gt_20pc,rv_r_gt_50pc,rv_r_gt_100pc,max_r_pc
target,,,,,,,
Hyades,1068,499,499,463,275,118,867.224140
Praesepe,522,142,142,131,85,34,179.833700
Stock 2,516,292,292,231,72,11,166.939966
IC 4756,135,67,67,55,19,1,186.509811


## 7. Spatial-separation sanity checks

Large cluster-centric separations can arise from genuine extended candidates,
distance uncertainties, catalogue contamination, or inconsistent distance
estimates.

Before interpreting the outer-tail counts dynamically, the distance and
separation distributions are audited at the unique-source level.

In [31]:
spatial_summary = []

for common_name, catalogue_name in target_aliases.items():

    sub = (
        tail_spatial[
            tail_spatial["Cluster"] == catalogue_name
        ]
        .sort_values("has_rv", ascending=False)
        .drop_duplicates("GaiaDR3")
    )

    spatial_summary.append({
        "target": common_name,
        "N": len(sub),
        "distance_median_pc": sub["rmedgeo"].median(),
        "distance_p05_pc": sub["rmedgeo"].quantile(0.05),
        "distance_p95_pc": sub["rmedgeo"].quantile(0.95),
        "r_median_pc": sub["r_cluster_pc"].median(),
        "r_p90_pc": sub["r_cluster_pc"].quantile(0.90),
        "r_p95_pc": sub["r_cluster_pc"].quantile(0.95),
        "r_p99_pc": sub["r_cluster_pc"].quantile(0.99),
        "r_max_pc": sub["r_cluster_pc"].max(),
    })

spatial_summary = pd.DataFrame(spatial_summary).set_index("target")

display(spatial_summary.round(2))

,N,distance_median_pc,distance_p05_pc,distance_p95_pc,r_median_pc,r_p90_pc,r_p95_pc,r_p99_pc,r_max_pc
target,,,,,,,,,
Hyades,1068,87.31,36.18,397.29,72.20,308.82,392.41,470.94,867.22
Praesepe,522,168.28,127.53,244.55,57.68,131.75,160.52,188.60,217.34
Stock 2,516,363.65,330.84,396.62,30.85,73.95,90.99,143.31,166.94
IC 4756,135,486.13,419.15,547.70,30.19,71.06,87.16,215.62,253.08


In [32]:
extreme_candidates = (
    tail_spatial[
        tail_spatial["Cluster"].isin(
            list(target_aliases.values())
        )
    ]
    .sort_values("r_cluster_pc", ascending=False)
    .drop_duplicates(["Cluster", "GaiaDR3"])
)

display(
    extreme_candidates[
        [
            "Cluster",
            "GaiaDR3",
            "Grade",
            "Class",
            "Ref",
            "RA_ICRS",
            "DE_ICRS",
            "rmedgeo",
            "RV",
            "r_cluster_pc",
        ]
    ].head(30)
)

,Cluster,GaiaDR3,Grade,Class,Ref,RA_ICRS,DE_ICRS,rmedgeo,RV,r_cluster_pc
5685,Melotte_25,3352156867126190208,G,T,Roser2019,98.846550,11.817967,907.1339,45.0119,867.224140
4622,Melotte_25,2000093736344536320,G,L,Jerabkova2021M1,335.057357,50.292137,503.7883,NaN,496.727792
3568,Melotte_25,5326922700281444864,G,T,Jerabkova2021M5,135.809516,-48.543716,495.3193,NaN,496.653196
3999,Melotte_25,5311109828992983168,G,T,Jerabkova2021M1,137.422092,-54.212643,486.2813,NaN,490.398980
4000,Melotte_25,5321783204981065600,G,T,Jerabkova2021M1,130.132921,-51.416068,490.0702,NaN,489.908527
3570,Melotte_25,5321706926366446720,G,T,Jerabkova2021M5,130.568654,-52.000183,480.4733,45.5291,480.792057
3998,Melotte_25,2171837663068365440,G,L,Jerabkova2021M5,324.274008,52.419672,482.4792,NaN,480.377989
3997,Melotte_25,2198408800352007296,G,L,Jerabkova2021M5,332.176258,56.537223,486.4535,NaN,479.767499
4001,Melotte_25,5310986619293965440,G,T,Jerabkova2021M1,137.270974,-54.768614,473.7481,9.5287,478.051131
4002,Melotte_25,5321706926366446336,G,T,Jerabkova2021M1,130.568724,-51.999410,471.0256,NaN,471.390514


In [33]:
tail_spatial["distance_ratio"] = (
    tail_spatial["rmedgeo"]
    / tail_spatial["cluster_distance"]
)

distance_ratio_summary = (
    tail_spatial[
        tail_spatial["Cluster"].isin(
            list(target_aliases.values())
        )
    ]
    .groupby("Cluster")["distance_ratio"]
    .describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    )
)

display(distance_ratio_summary)

,count,mean,std,min,1%,5%,50%,95%,99%,max
Cluster,,,,,,,,,,
IC_4756,135.0,1.034230,0.091448,0.870040,0.880995,0.898007,1.041519,1.173425,1.305893,1.400171
Melotte_25,2565.0,2.220876,1.903474,0.373288,0.573914,0.795683,1.633516,6.747523,9.738429,19.222742
NGC_2632,818.0,0.951822,0.193761,0.529496,0.601495,0.702092,0.915838,1.291357,1.472999,1.663999
Stock_2,676.0,0.988254,0.053567,0.848485,0.873071,0.903329,0.987901,1.069603,1.105504,1.332396


## 8. Reference-level consistency audit

Candidate samples are compared at the literature-reference level before
constructing the final analysis sample.

The purpose is to determine whether the apparent extended population is
supported independently by multiple catalogues or dominated by a single
membership selection.

In [34]:
reference_audit = []

for common_name, catalogue_name in target_aliases.items():

    sub = reliable_tail[
        reliable_tail["Cluster"] == catalogue_name
    ].copy()

    for ref, ref_sub in sub.groupby("Ref"):

        unique = ref_sub.drop_duplicates("GaiaDR3")

        reference_audit.append({
            "target": common_name,
            "ref": ref,
            "records": len(ref_sub),
            "unique_stars": unique["GaiaDR3"].nunique(),
            "rv_stars": unique.loc[
                unique["RV"].notna(), "GaiaDR3"
            ].nunique(),
            "leading": unique.loc[
                unique["Class"] == "L", "GaiaDR3"
            ].nunique(),
            "trailing": unique.loc[
                unique["Class"] == "T", "GaiaDR3"
            ].nunique(),
        })

reference_audit = pd.DataFrame(reference_audit)

display(
    reference_audit.sort_values(
        ["target", "unique_stars"],
        ascending=[True, False]
    )
)

,target,ref,records,unique_stars,rv_stars,leading,trailing
0,Hyades,Jerabkova2021M1,624,624,264,384,240
5,Hyades,Risbud2025,537,537,285,311,226
6,Hyades,Roser2019,448,448,263,252,196
4,Hyades,Oh2020,443,443,252,257,186
1,Hyades,Jerabkova2021M5,431,431,162,293,138
3,Hyades,Meingast2019,78,78,77,41,37
2,Hyades,Kos2024,4,4,4,0,4
11,IC 4756,Kos2024,135,135,67,88,47
7,Praesepe,Risbud2025,430,430,106,290,140
8,Praesepe,Roser2019,388,388,124,231,157


In [35]:
stock = reliable_tail[
    reliable_tail["Cluster"] == "Stock_2"
].copy()

stock_sets = {
    ref: set(group["GaiaDR3"].astype(str))
    for ref, group in stock.groupby("Ref")
}

refs = list(stock_sets)

overlap_rows = []

for i, ref1 in enumerate(refs):
    for ref2 in refs[i + 1:]:

        a = stock_sets[ref1]
        b = stock_sets[ref2]

        intersection = len(a & b)
        union = len(a | b)

        overlap_rows.append({
            "ref_1": ref1,
            "ref_2": ref2,
            "N_ref1": len(a),
            "N_ref2": len(b),
            "intersection": intersection,
            "jaccard": intersection / union if union else np.nan,
            "fraction_ref1_recovered": (
                intersection / len(a) if a else np.nan
            ),
            "fraction_ref2_recovered": (
                intersection / len(b) if b else np.nan
            ),
        })

stock_overlap = pd.DataFrame(overlap_rows)

display(stock_overlap)

,ref_1,ref_2,N_ref1,N_ref2,intersection,jaccard,fraction_ref1_recovered,fraction_ref2_recovered
0,Kos2024,Risbud2025,234,442,160,0.310078,0.683761,0.361991


In [36]:
stock_support = (
    stock.groupby("GaiaDR3")
    .agg(
        n_refs=("Ref", "nunique"),
        refs=("Ref", lambda x: ", ".join(sorted(set(x)))),
        has_rv=("RV", lambda x: x.notna().any()),
        classes=("Class", lambda x: ", ".join(sorted(set(x)))),
    )
)

print("Unique Stock 2 candidates:", len(stock_support))

print("\nReference support:")
print(stock_support["n_refs"].value_counts().sort_index())

print(
    "\nCandidates supported by >=2 references:",
    (stock_support["n_refs"] >= 2).sum()
)

print(
    "RV candidates supported by >=2 references:",
    (
        (stock_support["n_refs"] >= 2)
        & stock_support["has_rv"]
    ).sum()
)

display(
    stock_support[
        stock_support["n_refs"] >= 2
    ].head(30)
)

Unique Stock 2 candidates: 516

Reference support:
n_refs
1    356
2    160
Name: count, dtype: int64

Candidates supported by >=2 references: 160
RV candidates supported by >=2 references: 90


,n_refs,refs,has_rv,classes
GaiaDR3,,,,
430717054374342272,2,"Kos2024, Risbud2025",False,L
444102921240354816,2,"Kos2024, Risbud2025",False,T
446621352624952704,2,"Kos2024, Risbud2025",True,T
446651932791889664,2,"Kos2024, Risbud2025",True,T
447312258245108736,2,"Kos2024, Risbud2025",False,T
447518382313874560,2,"Kos2024, Risbud2025",True,T
447768761727525760,2,"Kos2024, Risbud2025",False,T
448223852163698176,2,"Kos2024, Risbud2025",True,T
454845218566439808,2,"Kos2024, Risbud2025",True,T


## 9. Cross-catalogue morphological agreement

For Stock 2 candidates independently recovered by Kos2024 and Risbud2025,
the leading/trailing classifications are compared directly.

Agreement in both membership and tail-side assignment provides a stronger
definition of the high-confidence morphological core than membership
intersection alone.

In [37]:
stock_class = (
    stock[
        ["GaiaDR3", "Ref", "Class"]
    ]
    .drop_duplicates()
)

class_matrix = (
    stock_class
    .pivot_table(
        index="GaiaDR3",
        columns="Ref",
        values="Class",
        aggfunc="first"
    )
)

display(class_matrix.head())

common_refs = class_matrix.dropna()

print("Stars with classifications from both references:", len(common_refs))

if {"Kos2024", "Risbud2025"}.issubset(common_refs.columns):

    agreement = (
        common_refs["Kos2024"]
        == common_refs["Risbud2025"]
    )

    print("Classification agreement:", agreement.sum())
    print("Classification disagreement:", (~agreement).sum())
    print("Agreement fraction:", agreement.mean())

    display(
        pd.crosstab(
            common_refs["Kos2024"],
            common_refs["Risbud2025"],
            margins=True
        )
    )

Ref,Kos2024,Risbud2025
GaiaDR3,,
245685533931916672,NaN,T
246306689281358976,NaN,T
249584917557569280,NaN,T
249631200124432128,NaN,T
250064820022590720,NaN,T


Stars with classifications from both references: 160
Classification agreement: 160
Classification disagreement: 0
Agreement fraction: 1.0


Risbud2025,L,T,All
Kos2024,,,
L,72,0,72
T,0,88,88
All,72,88,160


In [39]:
core_ids = set(
    stock_support[
        stock_support["n_refs"] >= 2
    ].index.astype(str)
)

stock_spatial = (
    tail_spatial[
        tail_spatial["Cluster"] == "Stock_2"
    ]
    .copy()
)

stock_spatial["GaiaDR3"] = stock_spatial["GaiaDR3"].astype(str)

stock_core = (
    stock_spatial[
        stock_spatial["GaiaDR3"].isin(core_ids)
    ]
    .sort_values("has_rv", ascending=False)
    .drop_duplicates("GaiaDR3")
)

print("Core sample:", len(stock_core))
print("Core with RV:", stock_core["RV"].notna().sum())

print("\n3D separation:")
print(
    stock_core["r_cluster_pc"].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print("\nOuter core:")
for radius in [10, 20, 30, 50, 75, 100]:
    print(
        f"> {radius:3d} pc:",
        (stock_core["r_cluster_pc"] > radius).sum()
    )

Core sample: 160
Core with RV: 90

3D separation:
count    160.000000
mean      24.012683
std       14.072461
min       10.794731
10%       12.574337
25%       14.604044
50%       19.823796
75%       29.244124
90%       38.481415
95%       54.302319
99%       77.070214
max       85.657874
Name: r_cluster_pc, dtype: float64

Outer core:
>  10 pc: 160
>  20 pc: 79
>  30 pc: 36
>  50 pc: 11
>  75 pc: 2
> 100 pc: 0


In [40]:
rv_core = stock_core[
    stock_core["RV"].notna()
].copy()

cluster_rv = (
    clusters_df.loc[
        clusters_df["Cluster"] == "Stock_2",
        "RV"
    ]
    .median()
)

rv_core["delta_rv"] = (
    rv_core["RV"] - cluster_rv
)

print("Stock 2 systemic RV:", cluster_rv)

print("\nCore RV residuals:")
print(
    rv_core["delta_rv"].describe(
        percentiles=[0.05, 0.1, 0.5, 0.9, 0.95]
    )
)

print("\n|ΔRV| thresholds:")

for dv in [1, 2, 3, 5, 10]:
    print(
        f"< {dv:2d} km/s:",
        (rv_core["delta_rv"].abs() < dv).sum()
    )

Stock 2 systemic RV: 8.16756427

Core RV residuals:
count    90.000000
mean     -0.140419
std       9.651932
min     -67.875364
5%       -7.214299
10%      -4.656714
50%       0.888136
90%       6.569356
95%       7.820276
max      22.172536
Name: delta_rv, dtype: float64

|ΔRV| thresholds:
<  1 km/s: 15
<  2 km/s: 36
<  3 km/s: 51
<  5 km/s: 66
< 10 km/s: 84


## 10. Feasibility decision

Stock 2 is adopted as the primary target for the subsequent analysis.

The catalogue audit identifies 516 unique Gold/Silver tail candidates in the
combined literature sample. Of these, 160 are recovered in both the Kos2024
and Risbud2025 reference subsets, with identical leading/trailing labels in
the current catalogue representation.

The cross-reference core remains spatially extended, with a median
cluster-centric separation of approximately 19.8 pc and candidates extending
to approximately 85.7 pc. Ninety core candidates have radial-velocity
measurements.

These results establish sample feasibility but do not by themselves establish
physical membership or tidal-tail origin. Subsequent analysis will therefore
audit catalogue provenance, Gaia astrometric quality, radial-velocity
uncertainties, and phase-space consistency before dynamical interpretation.

In [41]:
from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

reference_audit.to_csv(
    results_dir / "project00_reference_audit.csv",
    index=False
)

stock_overlap.to_csv(
    results_dir / "project00_stock2_reference_overlap.csv",
    index=False
)

stock_support.reset_index().to_csv(
    results_dir / "project00_stock2_support.csv",
    index=False
)

stock_core.to_csv(
    results_dir / "project00_stock2_core_feasibility.csv",
    index=False
)

outer_rv_inventory.to_csv(
    results_dir / "project00_outer_rv_inventory.csv"
)

print("Project 00 feasibility outputs saved.")

Project 00 feasibility outputs saved.
